# 02 - Amendment 3 Filter Creation

**Purpose:** Create filters to enable fair comparison of crash data before and after the June 16, 2025 reporting rule change

---

## The Problem: Apples vs Oranges

On June 16, 2025, NHTSA changed what crashes autonomous vehicle companies must report:

| Before June 16 | After June 16 |
|----------------|---------------|
| Report ALL crashes | Report only SERIOUS crashes |
| Even tiny fender-benders | Injuries, tow-aways, significant damage |
| ~1,200+ Waymo crashes | ~400 Waymo crashes |

If we compare raw crash counts before and after June 16, it looks like crashes dropped dramatically. But that's misleading - the rules just changed, not Waymo's safety.

**Our solution:** Create a filter that removes minor crashes from the pre-June 16 data, so we're comparing similar types of crashes.

---

## Two Filter Versions

We create two filters with different levels of strictness:

**V2 (Conservative):** Only includes crashes we're 100% certain would be reported under the new rules.

**V3 (With Delta-V Proxy):** Also includes crashes with significant impact force (delta-V >= 1 mph), which likely caused more than $1,000 in damage.

V3 is recommended for most analyses because it better matches the actual post-June 16 data patterns.

---

## Step 0: Import Libraries

In [ ]:
import pandas as pd
import os

## Step 1: Define File Paths

We'll load the merged data from the previous notebook.

In [ ]:
# =============================================================================
# FILE PATHS - Update the date when using new data
# =============================================================================

# Input: Merged data from notebook 01
PRIOR_DATA_PATH = "../data/waymo_crashes_prior_merged_20250129.csv"

# Output: Filtered datasets
OUTPUT_V2_PATH = "../data/waymo_crashes_v2_filtered_20250129.csv"
OUTPUT_V3_PATH = "../data/waymo_crashes_v3_filtered_20250129.csv"
OUTPUT_ALL_PATH = "../data/waymo_crashes_prior_with_filter_flags_20250129.csv"

# Verify input file exists
if os.path.exists(PRIOR_DATA_PATH):
    print(f"[OK] Input file found: {PRIOR_DATA_PATH}")
else:
    print(f"[MISSING] Input file not found: {PRIOR_DATA_PATH}")
    print("          Please run 01_data_loading_and_merging.ipynb first.")

## Step 2: Load the Pre-June 16 Data

We'll work with the pre-June 16 data since that's what we need to filter. The post-June 16 data is already filtered by NHTSA's new rules.

In [ ]:
# =============================================================================
# Load the pre-June 16 data
# =============================================================================

waymo_prior = pd.read_csv(PRIOR_DATA_PATH, dtype={'hub_zip_code': str, 'Zip Code': str})

print(f"Loaded pre-June 16 data")
print(f"  Crashes: {len(waymo_prior):,}")
print(f"  Columns: {len(waymo_prior.columns)}")
print()

# Check if we have the delta-V data we need
if 'hub_delta_v_less_than_1mph' in waymo_prior.columns:
    has_deltav = waymo_prior['hub_delta_v_less_than_1mph'].notna().sum()
    print(f"  Crashes with delta-V data: {has_deltav:,} ({100*has_deltav/len(waymo_prior):.1f}%)")
else:
    print("  [WARNING] Delta-V column not found. V3 filter will be limited.")

## Step 3: Understanding the New Reporting Rules

Under the new rules (after June 16, 2025), a crash must be reported if ANY of these are true:

### Always Report ("5-Day Triggers"):
1. **Any injury** - Someone was hurt (minor, moderate, serious, or fatal)
2. **Vulnerable road user hit** - Pedestrian, cyclist, or motorcyclist was involved
3. **Airbag deployed** - In any vehicle
4. **Vehicle towed** - Any vehicle had to be towed from the scene

### Property Damage Only - Report if:
5. **Damage over $1,000** - Significant property damage
6. **Single-vehicle crash** - Only the Waymo was involved (hit a pole, guardrail, etc.)
7. **Waymo struck another** - Waymo initiated the contact (not the other way around)

We can identify #1-4 and #6-7 directly from the data. For #5 (the $1,000 threshold), we use delta-V as a proxy: if impact force was less than 1 mph, damage was probably under $1,000.

## Step 4: Identify 5-Day Trigger Crashes

These crashes would ALWAYS be reported under the new rules, regardless of anything else.

In [ ]:
# =============================================================================
# CRITERION 1: Any injury reported
# =============================================================================

# The 'Highest Injury Severity Alleged' column tells us if anyone was hurt
# Values indicating injury: Minor, Moderate, Serious, Fatality
# Values indicating no injury: 'Property Damage. No Injured Reported', 'No Injured Reported'

injury_values = ['Minor', 'Moderate', 'Serious', 'Fatality']
has_injury = waymo_prior['Highest Injury Severity Alleged'].isin(injury_values)

print("CRITERION 1: Any injury")
print(f"  Crashes with injury: {has_injury.sum():,}")
print()
print("  Breakdown by severity:")
print(waymo_prior[has_injury]['Highest Injury Severity Alleged'].value_counts())

In [ ]:
# =============================================================================
# CRITERION 2: Vulnerable Road User (VRU) involved
# =============================================================================

# VRUs are pedestrians, cyclists, and motorcyclists
# They're more vulnerable than people in cars, so these crashes are always serious

vru_types = [
    'Non-Motorist: Pedestrian',  # Someone walking
    'Non-Motorist: Cyclist',     # Someone on a bicycle
    'Non-Motorist: Other',       # Other non-motorized (scooter, skateboard, etc.)
    'Motorcycle'                  # Motorcyclist
]

is_vru = waymo_prior['Crash With'].isin(vru_types)

print("CRITERION 2: Vulnerable Road User (VRU)")
print(f"  Crashes involving VRU: {is_vru.sum():,}")
print()
print("  Breakdown by type:")
print(waymo_prior[is_vru]['Crash With'].value_counts())

In [ ]:
# =============================================================================
# CRITERION 3: Airbag deployment (any vehicle)
# =============================================================================

# Check both the Subject Vehicle (SV = Waymo) and Crash Partner (CP = other vehicle)
# If either deployed airbags, it was a significant impact

sv_airbag = waymo_prior['SV Any Air Bags Deployed?'] == 'Yes'
cp_airbag = waymo_prior['CP Any Air Bags Deployed?'] == 'Yes'

# | means OR - true if either is true
has_airbag = sv_airbag | cp_airbag

print("CRITERION 3: Airbag deployment")
print(f"  Waymo airbag deployed: {sv_airbag.sum():,}")
print(f"  Other vehicle airbag:  {cp_airbag.sum():,}")
print(f"  Either (total):        {has_airbag.sum():,}")

In [ ]:
# =============================================================================
# CRITERION 4: Vehicle towed from scene
# =============================================================================

# If a vehicle had to be towed, it was too damaged to drive away
# Check both Waymo and the other vehicle

sv_towed = waymo_prior['SV Was Vehicle Towed?'] == 'Yes'
cp_towed = waymo_prior['CP Was Vehicle Towed?'] == 'Yes'

has_tow = sv_towed | cp_towed

print("CRITERION 4: Vehicle towed")
print(f"  Waymo towed:           {sv_towed.sum():,}")
print(f"  Other vehicle towed:   {cp_towed.sum():,}")
print(f"  Either (total):        {has_tow.sum():,}")

In [ ]:
# =============================================================================
# Combine all 5-day triggers
# =============================================================================

# A crash is a 5-day trigger if ANY of these is true
is_5day_trigger = has_injury | is_vru | has_airbag | has_tow

print("COMBINED 5-DAY TRIGGERS")
print(f"  Total crashes meeting ANY 5-day criterion: {is_5day_trigger.sum():,}")
print(f"  (Some crashes may meet multiple criteria)")
print()
print(f"  Remaining crashes (property damage only): {(~is_5day_trigger).sum():,}")

## Step 5: Filter Property Damage Only (PDO) Crashes

For crashes that don't meet any 5-day trigger, we need to check:
- Is damage over $1,000? (We use delta-V as a proxy)
- Was it a single-vehicle crash?
- Did Waymo strike the other vehicle?

In [ ]:
# =============================================================================
# CRITERION 6: Single-vehicle crash
# =============================================================================

# Single-vehicle means only Waymo was involved (no other cars)
# Examples: hitting a pole, guardrail, pothole, animal
# These are always reported because Waymo clearly caused them

single_vehicle_types = [
    'Other Fixed Object',  # Guardrail, barrier, etc.
    'Pole / Tree',         # Light pole, tree, etc.
    'Animal'               # Hit an animal
]

is_single_vehicle = waymo_prior['Crash With'].isin(single_vehicle_types)

print("CRITERION 6: Single-vehicle crash")
print(f"  Single-vehicle crashes: {is_single_vehicle.sum():,}")
print()
print("  Breakdown:")
print(waymo_prior[is_single_vehicle]['Crash With'].value_counts())

In [ ]:
# =============================================================================
# CRITERION 7: Waymo struck another vehicle
# =============================================================================

# The data doesn't directly tell us "who hit whom"
# But we can infer it from:
#   1. Was Waymo moving? (not stopped/parked)
#   2. Did Waymo have front-end damage? (implies Waymo hit something)
#   3. Did the other vehicle have rear damage? (implies Waymo rear-ended them)

# Step 1: Was Waymo moving?
stationary_movements = ['Stopped', 'Parked']
waymo_was_moving = ~waymo_prior['SV Pre-Crash Movement'].isin(stationary_movements)

print("Was Waymo moving?")
print(f"  Moving:     {waymo_was_moving.sum():,}")
print(f"  Stationary: {(~waymo_was_moving).sum():,}")

In [ ]:
# Step 2: Did Waymo have front-end contact?
# If Waymo hit something with its front, it probably struck the other vehicle

waymo_front_contact = (
    (waymo_prior['SV Contact Area - Front'] == 'Y') |
    (waymo_prior['SV Contact Area - Front Left'] == 'Y') |
    (waymo_prior['SV Contact Area - Front Right'] == 'Y')
)

print("Waymo contact area:")
print(f"  Front contact: {waymo_front_contact.sum():,}")

In [ ]:
# Step 3: Did the other vehicle have rear contact?
# This indicates Waymo rear-ended them

other_rear_contact = (
    (waymo_prior['CP Contact Area - Rear'] == 'Y') |
    (waymo_prior['CP Contact Area - Rear Left'] == 'Y') |
    (waymo_prior['CP Contact Area - Rear Right'] == 'Y')
)

print("Other vehicle contact area:")
print(f"  Rear contact: {other_rear_contact.sum():,}")

In [ ]:
# Combine: Waymo struck another if:
#   - Waymo was moving, AND
#   - (Waymo had front contact OR other vehicle had rear contact)

waymo_struck_another = waymo_was_moving & (waymo_front_contact | other_rear_contact)

print("CRITERION 7: Waymo struck another")
print(f"  Crashes where Waymo struck another: {waymo_struck_another.sum():,}")

In [ ]:
# =============================================================================
# CRITERION 5: Damage over $1,000 (using delta-V proxy)
# =============================================================================

# We don't have actual damage amounts, but we have delta-V:
#   Delta-V = change in velocity during impact
#   Lower delta-V = gentler impact = less damage
#
# Delta-V less than 1 mph is VERY minor (like bumping a shopping cart)
# It's very unlikely to cause $1,000+ in damage
#
# So: delta-V >= 1 mph is our proxy for "probably over $1,000 damage"

# hub_delta_v_less_than_1mph is True if delta-V was UNDER 1 mph
# We want crashes where it's False (delta-V was 1 mph or more)

deltav_over_1mph = waymo_prior['hub_delta_v_less_than_1mph'] == False

print("CRITERION 5: Significant damage (delta-V proxy)")
print(f"  Delta-V >= 1 mph (likely >$1,000 damage): {deltav_over_1mph.sum():,}")
print(f"  Delta-V < 1 mph (likely minor damage):   {(waymo_prior['hub_delta_v_less_than_1mph'] == True).sum():,}")
print(f"  Unknown delta-V:                         {waymo_prior['hub_delta_v_less_than_1mph'].isna().sum():,}")

## Step 6: Create the Filters

Now we combine our criteria into two filter versions:

**V2 (Conservative):** 5-day triggers + single-vehicle + Waymo struck another
- Does NOT use the delta-V proxy
- Only includes crashes we're CERTAIN would be reported

**V3 (Recommended):** Everything in V2 + crashes with delta-V >= 1 mph
- Uses delta-V as a proxy for the $1,000 damage threshold
- Better matches the actual post-June 16 data patterns

In [ ]:
# =============================================================================
# V2 FILTER (Conservative)
# =============================================================================

# Include crash if ANY of these is true:
#   - It's a 5-day trigger (injury, VRU, airbag, tow)
#   - It's a single-vehicle crash
#   - Waymo struck another vehicle

reportable_v2 = is_5day_trigger | is_single_vehicle | waymo_struck_another

# Add to dataframe
waymo_prior['reportable_v2'] = reportable_v2

print("V2 FILTER RESULTS (Conservative)")
print("="*50)
print(f"Total crashes:           {len(waymo_prior):,}")
print(f"Included (reportable):   {reportable_v2.sum():,}")
print(f"Excluded (minor):        {(~reportable_v2).sum():,}")
print(f"Reduction:               {100*(~reportable_v2).mean():.1f}%")

In [ ]:
# =============================================================================
# V3 FILTER (With Delta-V Proxy) - RECOMMENDED
# =============================================================================

# Include everything in V2, PLUS:
#   - Crashes with delta-V >= 1 mph (likely >$1,000 damage)

reportable_v3 = reportable_v2 | deltav_over_1mph

# Add to dataframe
waymo_prior['reportable_v3'] = reportable_v3

print("V3 FILTER RESULTS (With Delta-V Proxy)")
print("="*50)
print(f"Total crashes:           {len(waymo_prior):,}")
print(f"Included (reportable):   {reportable_v3.sum():,}")
print(f"Excluded (minor):        {(~reportable_v3).sum():,}")
print(f"Reduction:               {100*(~reportable_v3).mean():.1f}%")

In [ ]:
# =============================================================================
# Compare V2 and V3
# =============================================================================

print("FILTER COMPARISON")
print("="*60)
print()
print(f"{'Metric':<30} {'V2 (Conservative)':<15} {'V3 (Recommended)':<15}")
print("-"*60)
print(f"{'Included crashes':<30} {reportable_v2.sum():<15,} {reportable_v3.sum():<15,}")
print(f"{'Excluded crashes':<30} {(~reportable_v2).sum():<15,} {(~reportable_v3).sum():<15,}")
print(f"{'Reduction %':<30} {100*(~reportable_v2).mean():<15.1f} {100*(~reportable_v3).mean():<15.1f}")
print()

# How many differ?
diff = (reportable_v2 != reportable_v3).sum()
print(f"Crashes that differ between V2 and V3: {diff:,}")
print("(These are crashes included by V3 due to delta-V >= 1 mph)")

## Step 7: Add Detailed Filter Flags

For transparency, we'll add columns showing exactly WHY each crash was included or excluded.

In [ ]:
# =============================================================================
# Add filter decision flags (for transparency)
# =============================================================================

# These columns explain WHY each crash was included/excluded
# Prefix with 'filter_' so they're easy to identify

waymo_prior['filter_has_injury'] = has_injury
waymo_prior['filter_is_vru'] = is_vru
waymo_prior['filter_has_airbag'] = has_airbag
waymo_prior['filter_has_tow'] = has_tow
waymo_prior['filter_is_5day_trigger'] = is_5day_trigger
waymo_prior['filter_is_single_vehicle'] = is_single_vehicle
waymo_prior['filter_waymo_struck_another'] = waymo_struck_another
waymo_prior['filter_deltav_over_1mph'] = deltav_over_1mph
waymo_prior['filter_waymo_was_moving'] = waymo_was_moving
waymo_prior['filter_waymo_was_stationary'] = ~waymo_was_moving

print("Added filter decision columns:")
filter_cols = [col for col in waymo_prior.columns if col.startswith('filter_')]
for col in filter_cols:
    true_count = waymo_prior[col].sum()
    print(f"  {col}: {true_count:,} True")

## Step 8: Quality Check

Let's verify our filters make sense by checking what we're excluding.

In [ ]:
# =============================================================================
# Check what V3 is excluding
# =============================================================================

v3_excluded = waymo_prior[~waymo_prior['reportable_v3']]

print("V3 EXCLUDED CRASHES - Quality Check")
print("="*50)
print(f"Total excluded: {len(v3_excluded):,}")
print()

# These should all be 0 (if our filter is working correctly)
print("Sanity checks (should all be 0):")
print(f"  With injuries:         {v3_excluded['filter_has_injury'].sum()}")
print(f"  With tow:              {v3_excluded['filter_has_tow'].sum()}")
print(f"  With airbag:           {v3_excluded['filter_has_airbag'].sum()}")
print(f"  VRU involved:          {v3_excluded['filter_is_vru'].sum()}")
print()

# What are the excluded crashes like?
print("Characteristics of excluded crashes:")
stationary_pct = v3_excluded['filter_waymo_was_stationary'].mean()
low_deltav_pct = (v3_excluded['hub_delta_v_less_than_1mph'] == True).sum() / len(v3_excluded)

print(f"  Waymo was stationary:  {stationary_pct:.1%}")
print(f"  Delta-V < 1 mph:       {low_deltav_pct:.1%}")
print()
print("These are minor, other-party-at-fault crashes - exactly what we want to exclude.")

In [ ]:
# =============================================================================
# Check what V3 is including
# =============================================================================

v3_included = waymo_prior[waymo_prior['reportable_v3']]

print("V3 INCLUDED CRASHES - Breakdown by Reason")
print("="*50)
print(f"Total included: {len(v3_included):,}")
print()

# Count by primary reason (mutually exclusive assignment)
# Priority order: 5-day > single vehicle > Waymo struck > delta-V

n_5day = v3_included['filter_is_5day_trigger'].sum()
n_single = ((~v3_included['filter_is_5day_trigger']) & 
            v3_included['filter_is_single_vehicle']).sum()
n_struck = ((~v3_included['filter_is_5day_trigger']) & 
            (~v3_included['filter_is_single_vehicle']) & 
            v3_included['filter_waymo_struck_another']).sum()
n_deltav = ((~v3_included['filter_is_5day_trigger']) & 
            (~v3_included['filter_is_single_vehicle']) & 
            (~v3_included['filter_waymo_struck_another']) & 
            v3_included['filter_deltav_over_1mph']).sum()

print(f"  5-day triggers:      {n_5day:>5,} ({100*n_5day/len(v3_included):.1f}%)")
print(f"  Single-vehicle:      {n_single:>5,} ({100*n_single/len(v3_included):.1f}%)")
print(f"  Waymo struck:        {n_struck:>5,} ({100*n_struck/len(v3_included):.1f}%)")
print(f"  Delta-V >= 1 mph:    {n_deltav:>5,} ({100*n_deltav/len(v3_included):.1f}%)")
print(f"                       -----")
print(f"  Total:               {n_5day + n_single + n_struck + n_deltav:>5,}")

## Step 9: Save the Filtered Datasets

In [ ]:
# =============================================================================
# Save the output files
# =============================================================================

# 1. V2 filtered (conservative)
v2_filtered = waymo_prior[waymo_prior['reportable_v2']].copy()
v2_filtered.to_csv(OUTPUT_V2_PATH, index=False)
print(f"[OK] Saved V2 filtered: {OUTPUT_V2_PATH}")
print(f"     {len(v2_filtered):,} crashes")
print()

# 2. V3 filtered (recommended)
v3_filtered = waymo_prior[waymo_prior['reportable_v3']].copy()
v3_filtered.to_csv(OUTPUT_V3_PATH, index=False)
print(f"[OK] Saved V3 filtered: {OUTPUT_V3_PATH}")
print(f"     {len(v3_filtered):,} crashes")
print()

# 3. Complete dataset with all filter flags
waymo_prior.to_csv(OUTPUT_ALL_PATH, index=False)
print(f"[OK] Saved complete data with flags: {OUTPUT_ALL_PATH}")
print(f"     {len(waymo_prior):,} crashes, {len(waymo_prior.columns)} columns")

## Step 10: Summary

In [ ]:
# =============================================================================
# Final Summary
# =============================================================================

print("="*70)
print("FILTER CREATION COMPLETE")
print("="*70)
print()
print("WHAT WE CREATED:")
print()
print(f"Starting data: {len(waymo_prior):,} crashes (pre-June 16, all reported)")
print()
print(f"V2 Filter (Conservative):")
print(f"  Kept:     {reportable_v2.sum():,} crashes")
print(f"  Removed:  {(~reportable_v2).sum():,} crashes ({100*(~reportable_v2).mean():.0f}% reduction)")
print(f"  Use for:  Maximum certainty")
print()
print(f"V3 Filter (Recommended):")
print(f"  Kept:     {reportable_v3.sum():,} crashes")
print(f"  Removed:  {(~reportable_v3).sum():,} crashes ({100*(~reportable_v3).mean():.0f}% reduction)")
print(f"  Use for:  Better match to post-June 16 patterns")
print()
print("NEW COLUMNS ADDED:")
print("  - reportable_v2: True if crash passes V2 filter")
print("  - reportable_v3: True if crash passes V3 filter")
print("  - filter_*: Detailed flags showing why each crash was included/excluded")
print()
print("NEXT STEP:")
print("  Run 03_filter_validation_and_quality.ipynb to verify the filters")
print("  are working correctly by comparing to actual post-June 16 data.")